# Dependências

In [1]:
# Install required Google Cloud packages (commented out as these are typically one-time setup commands)
#!pip install gcloud
#!gcloud auth application-default login

# Import necessary Python libraries
import pandas as pd                # Data manipulation and analysis
import numpy as np                 # Numerical computing
import time                        # Time-related functions
import os                          # Operating system interfaces
import pandas_gbq                  # Pandas integration with BigQuery
from google.cloud import bigquery  # BigQuery client library
import glob                        # File path pattern matching
import openpyxl                    # Excel file handling
import csv                         # CSV file handling



c:\Users\ana.sales_republica\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Tratamento

In [2]:
diretorio = 'G:\\Drives compartilhados\\República.org\\4. Equipes\\Dados e Comunicação\\DADOS E CONHECIMENTO\\415 - Repositório de Dados\\Repositório Local\\PNAD\\2025'

In [3]:
os.chdir(diretorio)  

In [5]:
os.listdir(diretorio)

['pnad_indicadores.xlsx']

In [4]:
df = pd.read_excel('pnad_indicadores.xlsx', sheet_name='indicador_pnad_06')

In [6]:
df.columns

Index(['esfera', 'cor', 'sexo', 'freq', 'freq_se', 'freq_cv', 'prop',
       'prop_se', 'prop_cv'],
      dtype='object')

In [7]:
df["freq"] = df["freq"].round().astype("Int64")
df = df.drop(columns=['freq_se', 'freq_cv','prop_se', 'prop_cv'])
df["prop"] = (df["prop"] * 100).round(2)
df['ano'] = 2025
df = df.rename(columns= {'freq':'quantidade_vinculos', 'sexo':'genero','cor':'cor_raca'})   
df = df[['ano','esfera', 'cor_raca', 'genero','quantidade_vinculos', 'prop']]              
df

,ano,esfera,cor_raca,genero,quantidade_vinculos,prop
0,2025,Federal,Branca,Homem,537910,33.08
1,2025,Federal,Branca,Mulher,360222,22.16
2,2025,Federal,Negra,Homem,488829,30.07
3,2025,Federal,Negra,Mulher,203310,12.50
4,2025,Federal,Outra,Homem,17171,1.06
5,2025,Federal,Outra,Mulher,18431,1.13
6,2025,Estadual,Branca,Homem,806370,23.12
7,2025,Estadual,Branca,Mulher,900013,25.81
8,2025,Estadual,Negra,Homem,876241,25.13
9,2025,Estadual,Negra,Mulher,854745,24.51


In [96]:
df1 = pd.read_excel('G:\\Drives compartilhados\\República.org\\4. Equipes\\Dados e Comunicação\\DADOS E CONHECIMENTO\\415 - Repositório de Dados\\Repositório Local\\PNAD\\2024\\pnad_tabelas_graficos2024.xlsx', sheet_name='Dados6')
df1

,cor,sexo,esfera,sum,prop,cv,Unnamed: 6
0,Branca,Homem,Estadual,8.197671e+05,23.7,2.649866,0.236537
1,Branca,Mulher,Estadual,9.068609e+05,26.2,2.634666,0.261667
2,Negra,Homem,Estadual,8.675198e+05,25.0,2.646677,0.250316
3,Negra,Mulher,Estadual,8.269196e+05,23.9,2.536364,0.238601
4,Outra,Homem,Estadual,2.276095e+04,0.7,21.573643,0.006567
5,Outra,Mulher,Estadual,2.187346e+04,0.6,17.451043,0.006311
6,Branca,Homem,Federal,5.434920e+05,32.1,3.836764,0.321076
7,Branca,Mulher,Federal,3.671908e+05,21.7,4.813619,0.216924
8,Negra,Homem,Federal,5.152359e+05,30.4,3.655313,0.304384
9,Negra,Mulher,Federal,2.286794e+05,13.5,5.554415,0.135096


In [97]:
df1['ano'] = 2024
df1["sum"] = df1["sum"].round().astype("Int64")
df1 = df1.drop(columns=['cv', 'Unnamed: 6'])
df1 = df1.rename(columns= {'sum':'quantidade_vinculos', 'cor':'cor_raca','sexo':'genero'}) 
df1 = df1[['ano','esfera', 'cor_raca', 'genero','quantidade_vinculos', 'prop']]              
df1


,ano,esfera,cor_raca,genero,quantidade_vinculos,prop
0,2024,Estadual,Branca,Homem,819767,23.7
1,2024,Estadual,Branca,Mulher,906861,26.2
2,2024,Estadual,Negra,Homem,867520,25.0
3,2024,Estadual,Negra,Mulher,826920,23.9
4,2024,Estadual,Outra,Homem,22761,0.7
5,2024,Estadual,Outra,Mulher,21873,0.6
6,2024,Federal,Branca,Homem,543492,32.1
7,2024,Federal,Branca,Mulher,367191,21.7
8,2024,Federal,Negra,Homem,515236,30.4
9,2024,Federal,Negra,Mulher,228679,13.5


In [100]:
df_merged= pd.merge(df, df1, on=['ano', 'esfera', 'cor_raca', 'genero', 'quantidade_vinculos', 'prop'], how='outer')
df_merged

,ano,esfera,cor_raca,genero,quantidade_vinculos,prop
0,2024,Estadual,Branca,Homem,819767,23.70
1,2024,Estadual,Branca,Mulher,906861,26.20
2,2024,Estadual,Negra,Homem,867520,25.00
3,2024,Estadual,Negra,Mulher,826920,23.90
4,2024,Estadual,Outra,Homem,22761,0.70
5,2024,Estadual,Outra,Mulher,21873,0.60
6,2024,Federal,Branca,Homem,543492,32.10
7,2024,Federal,Branca,Mulher,367191,21.70
8,2024,Federal,Negra,Homem,515236,30.40
9,2024,Federal,Negra,Mulher,228679,13.50


# Upload

In [101]:
client = bigquery.Client(project='repositoriodedadosgpsp')

In [104]:
schema = [bigquery.SchemaField('ano', 'INTEGER', description= 'Ano de referência da observação'),
          bigquery.SchemaField('esfera', 'STRING', description= 'Nível da esfera do governo referente da observação'),
          bigquery.SchemaField('cor_raca', 'STRING', description= 'Raça/cor autodeclarado ou não'),
          bigquery.SchemaField('genero', 'STRING', description= 'Gênero autodeclarado ou não'),         
          bigquery.SchemaField('quantidade_vinculos', 'INTEGER', description= 'Número total de vinculos observados'),
          bigquery.SchemaField('prop', 'FLOAT', description= 'Proporção de vínculos em relação ao total naquele ano'),
          ]

dataset_ref = client.dataset('perfil_remuneracao')

table_ref = dataset_ref.table('PNAD_vinculos_genero_cor') 
job_config = bigquery.LoadJobConfig(schema=schema)
job = client.load_table_from_dataframe(df_merged, table_ref, job_config=job_config)
job.result()

LoadJob<project=repositoriodedadosgpsp, location=US, id=994c9731-4334-4523-8b23-f7882a4e2860>